# Data Prep for SVM

In [47]:
import polars as pl

## Train Test Split

Here we perform the split into train, validation and test data, with a **split** of 70% train, 15% validation and 15% test data. The split is performed **random**. 

In [48]:
SPATIAL_UNIT = "census" # option: census, community, hexa

In [ ]:
if SPATIAL_UNIT == "census":
    DATASET = "../data/processed_data/gold_hourly_demand_census_tracts.parquet"
elif SPATIAL_UNIT == "community":
    DATASET = "../data/processed_data/gold_hourly_demand_community_areas.parquet"
elif SPATIAL_UNIT == "hexa":
    DATASET = "../data/processed_data/gold_hourly_demand_hexagon.parquet"
else:
    print("Warning: No type of Spatial Data given, Used census tract")
    DATASET = "../data/processed_data/gold_hourly_demand_census_tracts.parquet"

OUTPUT = "../data/train_test_data/"
TARGET_COL = "trip_count"

SEED = 42
RANDOM = True

In [50]:
if(RANDOM == True):
    # split randomly
    df_split = (
        pl.scan_parquet(DATASET)
        .with_row_index("_row_id")
        .with_columns(
            (pl.col("_row_id").hash(seed=SEED) % 100).alias("_split_bucket")
        )
    )

    train = (
        df_split
        .filter(pl.col("_split_bucket") < 70)
        .drop(["_row_id", "_split_bucket"])
    )

    val = (
        df_split
        .filter(
            (pl.col("_split_bucket") >= 70) &
            (pl.col("_split_bucket") < 85)
        )
        .drop(["_row_id", "_split_bucket"])
    )

    test = (
        df_split
        .filter(pl.col("_split_bucket") >= 85)
        .drop(["_row_id", "_split_bucket"])
    )
else :
    # split according to time
    df_split = pl.scan_parquet(DATASET)

    train = df_split.filter(
        pl.col("datetime_hour") < pl.datetime(2025, 9, 1)
    )

    val = df_split.filter(
        (pl.col("datetime_hour") >= pl.datetime(2025, 9, 1)) &
        (pl.col("datetime_hour") < pl.datetime(2026, 1, 1))
    )

    test = df_split.filter(
        pl.col("datetime_hour") >= pl.datetime(2026, 1, 1)
    )


total_count = df_split.select(pl.len()).collect().item()
train_count = train.select(pl.len()).collect().item()
val_count = val.select(pl.len()).collect().item()
test_count = test.select(pl.len()).collect().item()

print("Total:", total_count)
print("Train:", train_count, " Share: ", round(train_count / total_count,2))
print("Val:", val_count, " Share: ", round(val_count / total_count,2))
print("Test:", test_count, " Share: ", round(test_count / total_count, 2))

train.sink_parquet(OUTPUT + "svm_" + SPATIAL_UNIT + "_train.parquet")
val.sink_parquet(OUTPUT + "svm_" + SPATIAL_UNIT + "_val.parquet")
test.sink_parquet(OUTPUT + "svm_" + SPATIAL_UNIT + "_test.parquet")

Total: 17930516
Train: 12551424  Share:  0.7
Val: 2687333  Share:  0.15
Test: 2691759  Share:  0.15


In [51]:
type(train)

polars.lazyframe.frame.LazyFrame

In [52]:
df_split.head(10).collect()

_row_id,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,p01m,skyc1_BKN,skyc1_CLR,skyc1_FEW,skyc1_OVC,skyc1_SCT,skyc1_VV,date,is_holiday,census_tract,food_drink,landmark,shop,train_station,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,_split_bucket
u32,datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,date,i8,i64,f64,f64,f64,f64,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,u64
0,2024-05-13 07:00:00,5,1,7,0.866025,-0.5,0.0,1.0,0.965926,-0.258819,20.0,50.59,10.0,10.0,0.0,0,0,1,0,0,0,2024-05-13,0,17031292300,0.0,1.0,0.0,5.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",55
1,2024-05-06 02:00:00,5,1,2,0.866025,-0.5,0.0,1.0,0.5,0.866025,10.0,71.07,3.0,10.0,0.0,1,0,0,0,0,0,2024-05-06,0,17031292300,0.0,1.0,0.0,5.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",30
2,2024-05-25 05:00:00,5,6,5,0.866025,-0.5,-0.974928,-0.222521,0.965926,0.258819,15.0,77.55,9.0,10.0,0.0,0,0,1,0,0,0,2024-05-25,0,17031292300,0.0,1.0,0.0,5.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",67
3,2024-06-04 12:00:00,6,2,12,0.5,-0.866025,0.781831,0.62349,1.2246e-16,-1.0,30.0,44.65,9.0,10.0,0.0,0,0,0,0,1,0,2024-06-04,0,17031292300,0.0,1.0,0.0,5.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",81
4,2024-04-25 14:00:00,4,4,14,1.0,6.1232e-17,0.433884,-0.900969,-0.5,-0.866025,13.89,43.46,12.0,10.0,0.0,0,0,1,0,0,0,2024-04-25,0,17031292300,0.0,1.0,0.0,5.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",82
5,2024-06-02 02:00:00,6,7,2,0.5,-0.866025,-0.781831,0.62349,0.5,0.866025,15.56,93.1,6.0,3.0,0.0,0,0,0,1,0,0,2024-06-02,0,17031292300,0.0,1.0,0.0,5.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",13
6,2024-06-02 18:00:00,6,7,18,0.5,-0.866025,-0.781831,0.62349,-1.0,-1.8370e-16,21.67,56.96,7.0,10.0,0.0,0,0,1,0,0,0,2024-06-02,0,17031292300,0.0,1.0,0.0,5.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",50
7,2024-06-05 02:00:00,6,3,2,0.5,-0.866025,0.974928,-0.222521,0.5,0.866025,20.56,90.17,9.5,6.5,11.43,0,0,0,0,1,0,2024-06-05,0,17031292300,0.0,1.0,0.0,5.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",1
8,2024-05-10 13:00:00,5,5,13,0.866025,-0.5,-0.433884,-0.900969,-0.258819,-0.965926,17.78,51.93,10.0,10.0,0.0,0,0,0,0,1,0,2024-05-10,0,17031292300,0.0,1.0,0.0,5.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",29
